[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/02_sizing_and_serving/02.1_capacity_planning/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/02_sizing_and_serving/02.1_capacity_planning/lab.ipynb)

# Lab 2.1: Capacity Planning Calculator

Pick your model and see the memory math step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# Model specifications: params in billions, layer/head/dim counts
MODELS = {
    'Mistral-7B':  {'params_B': 7.24,  'layers': 32,  'heads': 32,  'head_dim': 128},
    'Llama-70B':   {'params_B': 70.6,  'layers': 80,  'heads': 64,  'head_dim': 128},
    'Llama-405B':  {'params_B': 405.0, 'layers': 126, 'heads': 128, 'head_dim': 128},
}

# GPU VRAM in GB
GPUS = {
    'T4': 16, 'A10G': 24, 'A100-80': 80, 'H100': 80, 'H200': 141,
}

# Bytes per parameter for each precision
PRECISIONS = {
    'FP16': 2, 'INT8': 1, 'INT4': 0.5,
}

# Approximate cost per hour (USD) for reference
GPU_COST = {
    'T4': 0.53, 'A10G': 1.21, 'A100-80': 3.67, 'H100': 4.25, 'H200': 5.50,
}

print('Models:', list(MODELS.keys()))
print('GPUs:', list(GPUS.keys()))
print('Precisions:', list(PRECISIONS.keys()))

## Step-by-Step Capacity Calculator

In [ ]:
def capacity_calc(model_name, gpu_name, precision, tokens_per_conversation, num_users):
    """Show every calculation step for GPU memory capacity planning."""
    m = MODELS[model_name]
    gpu_vram = GPUS[gpu_name]
    bpp = PRECISIONS[precision]  # bytes per parameter

    # Step 1: Weight memory
    weight_gb = m['params_B'] * bpp  # billions of params * bytes = GB
    print(f'Step 1: Weight Memory')
    print(f'  params x bytes_per_param = {m["params_B"]}B x {bpp} = {weight_gb:.1f} GB')
    print()

    # Step 2: KV cache per token (bytes, then convert)
    # Formula: 2 (K+V) x heads x head_dim x layers x bytes_per_value
    kv_bytes_per_token = 2 * m['heads'] * m['head_dim'] * m['layers'] * bpp
    kv_mb_per_token = kv_bytes_per_token / (1024**2)
    print(f'Step 2: KV Cache Per Token')
    print(f'  2 (K+V) x {m["heads"]} heads x {m["head_dim"]} dim x {m["layers"]} layers x {bpp} bytes')
    print(f'  = {kv_bytes_per_token:,.0f} bytes = {kv_mb_per_token:.2f} MB/token')
    print()

    # Step 3: KV cache per user
    kv_per_user_gb = (kv_bytes_per_token * tokens_per_conversation) / (1024**3)
    print(f'Step 3: KV Cache Per User')
    print(f'  {kv_mb_per_token:.2f} MB/token x {tokens_per_conversation} tokens = {kv_per_user_gb:.2f} GB/user')
    print()

    # Step 4: Total KV cache for all users
    total_kv_gb = kv_per_user_gb * num_users
    print(f'Step 4: Total KV Cache')
    print(f'  {kv_per_user_gb:.2f} GB/user x {num_users} users = {total_kv_gb:.1f} GB')
    print()

    # Step 5: Total memory (weights + KV + 10% overhead)
    overhead_gb = weight_gb * 0.10  # 10% of weights for activations/framework
    total_gb = weight_gb + total_kv_gb + overhead_gb
    print(f'Step 5: Total Memory Needed')
    print(f'  weights + total_kv + overhead(10%) = {weight_gb:.1f} + {total_kv_gb:.1f} + {overhead_gb:.1f} = {total_gb:.1f} GB')
    print()

    # Step 6: Fit check
    fits = total_gb <= gpu_vram
    headroom = gpu_vram - total_gb
    print(f'Step 6: Does it fit?')
    print(f'  GPU VRAM = {gpu_vram} GB')
    print(f'  Needed   = {total_gb:.1f} GB')
    if fits:
        print(f'  \U00002705 FITS ({headroom:.1f} GB headroom)')
    else:
        print(f'  \U0000274C OOM (need {-headroom:.1f} more GB)')
    print()

    # Stacked bar chart: weights | KV | overhead vs GPU VRAM
    fig, ax = plt.subplots(figsize=(8, 2))
    ax.barh(['Memory'], [weight_gb], color='#dbeafe', edgecolor='#000', label='Weights')
    ax.barh(['Memory'], [total_kv_gb], left=[weight_gb], color='#fef3c7', edgecolor='#000', label='KV Cache')
    ax.barh(['Memory'], [overhead_gb], left=[weight_gb + total_kv_gb], color='#f3e8ff', edgecolor='#000', label='Overhead')
    # GPU VRAM line
    ax.axvline(gpu_vram, color='red', linestyle='--', linewidth=2, label=f'{gpu_name} VRAM ({gpu_vram} GB)')
    ax.set_xlabel('GB')
    ax.set_title('Memory Breakdown vs GPU Capacity')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_xlim(0, max(total_gb, gpu_vram) * 1.1)
    plt.tight_layout()
    plt.show()

# Build interactive widgets
widgets.interact(
    capacity_calc,
    model_name=widgets.Dropdown(options=list(MODELS.keys()), value='Mistral-7B', description='Model:'),
    gpu_name=widgets.Dropdown(options=list(GPUS.keys()), value='A100-80', description='GPU:'),
    precision=widgets.Dropdown(options=list(PRECISIONS.keys()), value='FP16', description='Precision:'),
    tokens_per_conversation=widgets.IntSlider(min=256, max=32768, step=256, value=2048, description='Tokens/conv:',
                                              style={'description_width': 'initial'}),
    num_users=widgets.IntSlider(min=1, max=256, step=1, value=16, description='Concurrent users:',
                                style={'description_width': 'initial'}),
);

## What's the Maximum Users?

In [ ]:
def max_users_calc(model_name, gpu_name, precision, tokens_per_conversation):
    """Compute max concurrent users that fit in GPU memory."""
    m = MODELS[model_name]
    gpu_vram = GPUS[gpu_name]
    bpp = PRECISIONS[precision]

    # Weight memory + overhead
    weight_gb = m['params_B'] * bpp
    overhead_gb = weight_gb * 0.10
    available_gb = gpu_vram - weight_gb - overhead_gb

    # KV cache per user
    kv_bytes_per_token = 2 * m['heads'] * m['head_dim'] * m['layers'] * bpp
    kv_per_user_gb = (kv_bytes_per_token * tokens_per_conversation) / (1024**3)

    # Max users
    if available_gb <= 0 or kv_per_user_gb <= 0:
        max_users = 0
    else:
        max_users = int(available_gb / kv_per_user_gb)

    print(f'Available for KV = {gpu_vram} GB - {weight_gb:.1f} GB (weights) - {overhead_gb:.1f} GB (overhead) = {available_gb:.1f} GB')
    print(f'KV per user = {kv_per_user_gb:.3f} GB')
    print(f'Max concurrent users = {available_gb:.1f} / {kv_per_user_gb:.3f} = {max_users} users')
    print()

    # Horizontal bar: used vs available
    used_gb = weight_gb + overhead_gb + (kv_per_user_gb * min(max_users, 256))
    fig, ax = plt.subplots(figsize=(8, 1.5))
    ax.barh(['VRAM'], [weight_gb + overhead_gb], color='#dbeafe', edgecolor='#000', label='Weights+Overhead')
    ax.barh(['VRAM'], [kv_per_user_gb * max_users], left=[weight_gb + overhead_gb],
            color='#dcfce7', edgecolor='#000', label=f'KV ({max_users} users)')
    ax.axvline(gpu_vram, color='red', linestyle='--', linewidth=2, label=f'VRAM limit')
    ax.set_xlabel('GB')
    ax.set_xlim(0, gpu_vram * 1.1)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

widgets.interact(
    max_users_calc,
    model_name=widgets.Dropdown(options=list(MODELS.keys()), value='Mistral-7B', description='Model:'),
    gpu_name=widgets.Dropdown(options=list(GPUS.keys()), value='A100-80', description='GPU:'),
    precision=widgets.Dropdown(options=list(PRECISIONS.keys()), value='FP16', description='Precision:'),
    tokens_per_conversation=widgets.IntSlider(min=256, max=32768, step=256, value=2048, description='Tokens/conv:',
                                              style={'description_width': 'initial'}),
);

## Try Different Scenarios

In [ ]:
def gpu_comparison(model_name, precision, tokens_per_conversation):
    """Compare all GPUs for the selected model configuration."""
    m = MODELS[model_name]
    bpp = PRECISIONS[precision]
    weight_gb = m['params_B'] * bpp
    overhead_gb = weight_gb * 0.10
    kv_bytes_per_token = 2 * m['heads'] * m['head_dim'] * m['layers'] * bpp
    kv_per_user_gb = (kv_bytes_per_token * tokens_per_conversation) / (1024**3)

    print(f'Model: {model_name} | Precision: {precision} | Tokens/conv: {tokens_per_conversation}')
    print(f'Weight memory: {weight_gb:.1f} GB | KV/user: {kv_per_user_gb:.3f} GB')
    print()
    # Table header
    print(f'{"GPU":<10} {"VRAM":<8} {"Weights Fit?":<14} {"Max Users":<12} {"$/hr":<8}')
    print('-' * 52)

    for gpu_name, vram in GPUS.items():
        # Check if weights alone fit
        weights_fit = weight_gb + overhead_gb <= vram
        fit_str = '\U00002705 Yes' if weights_fit else '\U0000274C No'
        # Max users
        available = vram - weight_gb - overhead_gb
        max_u = int(available / kv_per_user_gb) if (available > 0 and kv_per_user_gb > 0) else 0
        max_u = max(max_u, 0)
        cost = GPU_COST.get(gpu_name, 0)
        print(f'{gpu_name:<10} {vram:<8} {fit_str:<14} {max_u:<12} ${cost:.2f}')

widgets.interact(
    gpu_comparison,
    model_name=widgets.Dropdown(options=list(MODELS.keys()), value='Mistral-7B', description='Model:'),
    precision=widgets.Dropdown(options=list(PRECISIONS.keys()), value='FP16', description='Precision:'),
    tokens_per_conversation=widgets.IntSlider(min=256, max=32768, step=256, value=2048, description='Tokens/conv:',
                                              style={'description_width': 'initial'}),
);

## Key Takeaway

**KV cache is the variable cost.** Longer conversations or more concurrent users = more KV cache memory = fewer users fit on one GPU.

The model weights are a fixed cost (same regardless of load). The KV cache grows linearly with both sequence length and batch size. This is why capacity planning for LLM serving is fundamentally about managing KV cache memory.